# Class 11 — RL Environments for LLMs

> *"Environments are synthetic data engines, RL trainers, and eval harnesses — all the same artifact."*  

You've spent the last few classes learning how to *train* models — CPT, SFT, DPO, RLVR. But every reward signal you used lived inside an **environment**, even if we never named it. This class names it, builds it from scratch, then shows you the modern tooling (Verifiers + Prime Intellect) that the open-source community is using to train frontier-scale models like INTELLECT-3.

By the end of class you will have:

1. A clear mental model of what an RL environment actually *is* — three pieces, no magic.
2. A working coding environment **built from scratch** in ~50 lines of Python (no library).
3. The same environment **rebuilt with Verifiers**, showing what the abstraction buys you.
4. A **multi-turn iterative** version where the model writes code, sees test failures, and fixes them.
5. A **cross-model comparison** — same environment, different models, ranked by pass rate.

Why this matters: when you call OpenRouter or train a model with GRPO, you're using an environment. The closed labs are spending billions building proprietary environments. The open-source community is fighting back with the [Environments Hub](https://app.primeintellect.ai/dashboard/environments). Knowing how to build one is the difference between using AI and *building* AI infrastructure.


---

## 0. Setup

We need:
- **OpenRouter API key** for model inference — sign up at [openrouter.ai](https://openrouter.ai/) (free credits available)
- **`openai` SDK** to talk to OpenRouter (it's OpenAI-compatible)
- **`datasets`** for HuggingFace dataset format (used later by Verifiers)

We'll install Verifiers later, in the section where we actually use it.


In [1]:
# Install dependencies
!pip install -q openai datasets


In [3]:
import os
import json
import subprocess
import tempfile
import textwrap
from google.colab import userdata
from typing import Any, Callable



from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get('OPENROUTER_API_KEY'),
)

# Quick sanity check
resp = client.chat.completions.create(
    model="google/gemini-2.0-flash-001",
    messages=[{"role": "user", "content": "Reply with exactly the word 'pong'."}],
    max_tokens=10,
)
print("Model said:", resp.choices[0].message.content.strip())


Model said: pong


---

## 1. The mental model

An RL environment for LLMs is **three things in a trench coat**:

```
┌───────────────────────────────────────────────────┐
│  ENVIRONMENT                                      │
│                                                   │
│   1. Dataset      — task inputs (prompts)         │
│   2. Rollout      — how the model interacts       │
│                     (single call? tool use?       │
│                      multi-turn? sandbox?)        │
│   3. Rubric       — scoring function              │
│                     (verifies the output)         │
│                                                   │
└───────────────────────────────────────────────────┘
```

That's it. No magic. The same artifact is used for:
- **Evaluation** — run rollouts, average rubric scores → benchmark number
- **RL training** — same loop, but use rubric scores as rewards to update the model
- **Synthetic data generation** — keep the high-reward rollouts as SFT data
- **Agent harness experimentation** — swap models or scaffolds, see what changes

That's the Will Brown insight: **environments unify the entire post-training stack**. Once you can build one, everything else is just "what do I do with the rollouts?"

We'll prove this by building one, three ways.


---

## 2. The dataset

For a coding environment, each "task" is a problem statement plus a way to verify the solution. We'll use the simplest possible structure: a prompt, a function name, and a list of test cases (input expression → expected output).

In production you'd use HumanEval, MBPP, LiveCodeBench, or SWE-bench. Here we hand-roll five toy problems so we can see every step clearly.


In [4]:
PROBLEMS = [
    {
        "id": "add",
        "prompt": "Write a Python function `add(a, b)` that returns a + b.",
        "function_name": "add",
        "tests": [
            ("add(1, 2)", 3),
            ("add(-5, 5)", 0),
            ("add(0, 0)", 0),
            ("add(100, 200)", 300),
        ],
    },
    {
        "id": "is_even",
        "prompt": "Write a Python function `is_even(n)` that returns True if n is even, False otherwise.",
        "function_name": "is_even",
        "tests": [
            ("is_even(2)", True),
            ("is_even(3)", False),
            ("is_even(0)", True),
            ("is_even(-4)", True),
        ],
    },
    {
        "id": "reverse_string",
        "prompt": "Write a Python function `reverse_string(s)` that returns the reverse of string s.",
        "function_name": "reverse_string",
        "tests": [
            ("reverse_string('hello')", "olleh"),
            ("reverse_string('')", ""),
            ("reverse_string('a')", "a"),
            ("reverse_string('ab')", "ba"),
        ],
    },
    {
        "id": "fizzbuzz",
        "prompt": (
            "Write a Python function `fizzbuzz(n)` that returns 'Fizz' if n is divisible by 3, "
            "'Buzz' if divisible by 5, 'FizzBuzz' if divisible by both, else str(n)."
        ),
        "function_name": "fizzbuzz",
        "tests": [
            ("fizzbuzz(3)", "Fizz"),
            ("fizzbuzz(5)", "Buzz"),
            ("fizzbuzz(15)", "FizzBuzz"),
            ("fizzbuzz(7)", "7"),
        ],
    },
    {
        "id": "count_vowels",
        "prompt": "Write a Python function `count_vowels(s)` that returns the count of vowels (aeiouAEIOU) in s.",
        "function_name": "count_vowels",
        "tests": [
            ("count_vowels('hello')", 2),
            ("count_vowels('xyz')", 0),
            ("count_vowels('AEIOU')", 5),
            ("count_vowels('')", 0),
        ],
    },
]

print(f"Loaded {len(PROBLEMS)} problems.")
for p in PROBLEMS:
    print(f"  - {p['id']}: {len(p['tests'])} tests")


Loaded 5 problems.
  - add: 4 tests
  - is_even: 4 tests
  - reverse_string: 4 tests
  - fizzbuzz: 4 tests
  - count_vowels: 4 tests


---

## 3. The sandbox

When the model writes code, we need to **execute it safely**. In production this means Docker containers, [Prime Sandboxes](https://www.primeintellect.ai/), [E2B](https://e2b.dev/), or similar. Here we'll use the simplest thing that works on Colab: `subprocess.run` with a timeout.

> ⚠️ **This is not production-secure.** A subprocess on the same VM can do plenty of damage if the code is malicious. We're using it because the *concept* of sandboxing is what matters — the production hardening is an infra detail you'd outsource to a sandbox-as-a-service provider.

Three things our sandbox needs to handle:
1. **Syntax errors** — model writes code that doesn't parse
2. **Timeouts** — model writes infinite loops
3. **Test failures** — code runs but returns wrong answer


In [5]:
def run_code(code: str, test_call: str, expected: Any, timeout: int = 5) -> dict:
    """Execute code in a subprocess, run one test, return result.

    Returns dict with: passed (bool), error (str or None), output (str).
    """
    # Build a script that runs the user code, executes the test, and prints result.
    # We use repr() for the expected value so strings/bools/numbers all work.
    script = textwrap.dedent(f"""
        import sys
        try:
{textwrap.indent(code, ' ' * 12)}
        except Exception as e:
            print(f"SETUP_ERROR: {{type(e).__name__}}: {{e}}")
            sys.exit(1)

        try:
            result = {test_call}
            expected = {expected!r}
            if result == expected:
                print("PASS")
            else:
                print(f"FAIL: got {{result!r}}, expected {{expected!r}}")
        except Exception as e:
            print(f"RUNTIME_ERROR: {{type(e).__name__}}: {{e}}")
    """)

    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(script)
        path = f.name

    try:
        proc = subprocess.run(
            ['python', path],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        output = proc.stdout.strip() + proc.stderr.strip()
        passed = output.startswith("PASS")
        error = None if passed else output
        return {"passed": passed, "error": error, "output": output}
    except subprocess.TimeoutExpired:
        return {"passed": False, "error": "TIMEOUT", "output": ""}
    finally:
        os.unlink(path)


Let's stress-test the sandbox before we trust it. Three failure modes — does it handle them all?

In [6]:
# Case 1: correct code
print("Case 1 — correct code:")
print(run_code("def add(a, b):\n    return a + b", "add(2, 3)", 5))

# Case 2: wrong answer
print("\nCase 2 — wrong answer:")
print(run_code("def add(a, b):\n    return a - b", "add(2, 3)", 5))

# Case 3: syntax error
print("\nCase 3 — syntax error:")
print(run_code("def add(a, b\n    return a + b", "add(2, 3)", 5))

# Case 4: infinite loop (should timeout)
print("\nCase 4 — infinite loop:")
print(run_code("def add(a, b):\n    while True: pass\n    return a + b", "add(2, 3)", 5, timeout=2))


Case 1 — correct code:
{'passed': True, 'error': None, 'output': 'PASS'}

Case 2 — wrong answer:
{'passed': False, 'error': 'FAIL: got -1, expected 5', 'output': 'FAIL: got -1, expected 5'}

Case 3 — syntax error:
{'passed': False, 'error': 'File "/tmp/tmp54_thu_l.py", line 4\n    def add(a, b\n           ^\nSyntaxError: \'(\' was never closed', 'output': 'File "/tmp/tmp54_thu_l.py", line 4\n    def add(a, b\n           ^\nSyntaxError: \'(\' was never closed'}

Case 4 — infinite loop:
{'passed': False, 'error': 'TIMEOUT', 'output': ''}


---

## 4. The environment, from scratch

Now we have all three pieces. Let's wire them together — this is the entire environment, no library:

```
PROBLEMS  ──┐
            ├──►  ROLLOUT  ──►  generated_code  ──►  RUBRIC  ──►  score
   model  ──┘                                          │
                                                       ▼
                                                  pass rate
```

The whole thing fits in one screen.


In [7]:
def rollout(problem: dict, model: str = "google/gemini-2.0-flash-001") -> str:
    """Ask the model to solve the problem. Return the raw response text."""
    system = (
        "You are a Python coding assistant. "
        "When asked to write a function, respond with ONLY the function definition. "
        "No explanations, no markdown fences, just the code."
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": problem["prompt"]},
        ],
        temperature=0.0,
        max_tokens=300,
    )
    return resp.choices[0].message.content


def rubric(generated_code: str, problem: dict) -> dict:
    """Score generated code against problem's tests. Return per-test results + pass rate."""
    results = []
    for test_call, expected in problem["tests"]:
        result = run_code(generated_code, test_call, expected)
        results.append((test_call, result["passed"], result.get("error")))
    pass_rate = sum(1 for _, passed, _ in results if passed) / len(results)
    return {"pass_rate": pass_rate, "results": results}


def evaluate(problems: list[dict], model: str) -> dict:
    """Run the full rollout-and-score loop over all problems. This IS the environment."""
    scores = []
    transcripts = []
    for p in problems:
        code = rollout(p, model=model)
        score = rubric(code, p)
        scores.append(score["pass_rate"])
        transcripts.append({
            "problem": p["id"],
            "code": code,
            "pass_rate": score["pass_rate"],
            "details": score["results"],
        })
        print(f"  {p['id']}: {score['pass_rate']:.0%} ({sum(1 for _, ok, _ in score['results'] if ok)}/{len(score['results'])} tests)")
    return {
        "mean_pass_rate": sum(scores) / len(scores),
        "transcripts": transcripts,
    }


In [8]:
# Run the environment
print("Evaluating gemini-2.0-flash on our 5 problems:")
print("-" * 50)
result = evaluate(PROBLEMS, model="google/gemini-2.0-flash-001")
print("-" * 50)
print(f"\nOverall pass rate: {result['mean_pass_rate']:.1%}")


Evaluating gemini-2.0-flash on our 5 problems:
--------------------------------------------------
  add: 0% (0/4 tests)
  is_even: 0% (0/4 tests)
  reverse_string: 0% (0/4 tests)
  fizzbuzz: 0% (0/4 tests)
  count_vowels: 0% (0/4 tests)
--------------------------------------------------

Overall pass rate: 0.0%


**That's an environment.** ~50 lines of code, no library, no magic. Three pieces: dataset, rollout, rubric. The "evaluate" function is the loop that ties them together.

Take a moment to inspect a transcript to see exactly what the model produced:

In [9]:
# Look at one transcript in full
import pprint
pprint.pp(result["transcripts"][0])


{'problem': 'add',
 'code': '```python\ndef add(a, b):\n  return a + b\n```',
 'pass_rate': 0.0,
 'details': [('add(1, 2)',
              False,
              'File "/tmp/tmphp33e4gu.py", line 4\n'
              '    ```python\n'
              '    ^\n'
              'SyntaxError: invalid syntax'),
             ('add(-5, 5)',
              False,
              'File "/tmp/tmp6so_82oc.py", line 4\n'
              '    ```python\n'
              '    ^\n'
              'SyntaxError: invalid syntax'),
             ('add(0, 0)',
              False,
              'File "/tmp/tmp7pri6_gv.py", line 4\n'
              '    ```python\n'
              '    ^\n'
              'SyntaxError: invalid syntax'),
             ('add(100, 200)',
              False,
              'File "/tmp/tmp8m4e4pbn.py", line 4\n'
              '    ```python\n'
              '    ^\n'
              'SyntaxError: invalid syntax')]}


---

## 5. Break it

The environment "works" but it's brittle. What happens if the model wraps its answer in a markdown code fence? Or includes prose alongside the function? Or returns extra explanation? The system prompt asked it not to, but models don't always obey.

Let's deliberately provoke failure to see what bad rubric design looks like.


In [10]:
# Force the model to be chatty by removing the strict system prompt
def chatty_rollout(problem: dict, model: str = "google/gemini-2.0-flash-001") -> str:
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": problem["prompt"]}],
        temperature=0.0,
        max_tokens=300,
    )
    return resp.choices[0].message.content

# Run on one problem
chatty_code = chatty_rollout(PROBLEMS[0])
print("Model output:\n")
print(chatty_code)
print("\n" + "=" * 50)
print("Score with naive rubric:")
print(rubric(chatty_code, PROBLEMS[0]))


Model output:

```python
def add(a, b):
  """
  This function takes two numbers as input and returns their sum.

  Args:
    a: The first number.
    b: The second number.

  Returns:
    The sum of a and b.
  """
  return a + b
```

Score with naive rubric:
{'pass_rate': 0.0, 'results': [('add(1, 2)', False, 'File "/tmp/tmpl1_oln7h.py", line 4\n    ```python\n    ^\nSyntaxError: invalid syntax'), ('add(-5, 5)', False, 'File "/tmp/tmpwqd0ueg9.py", line 4\n    ```python\n    ^\nSyntaxError: invalid syntax'), ('add(0, 0)', False, 'File "/tmp/tmpot7l5h1j.py", line 4\n    ```python\n    ^\nSyntaxError: invalid syntax'), ('add(100, 200)', False, 'File "/tmp/tmpx6bcu4jg.py", line 4\n    ```python\n    ^\nSyntaxError: invalid syntax')]}


You probably see what went wrong. Markdown fences (` ```python ... ``` `) make Python parse the whole blob as a syntax error. The model is *correct*, but our rubric scored it 0.

This is **rubric brittleness**, the most common bug in environment design. The fix is to extract code more robustly before evaluating it.

In [11]:
import re

def extract_code(text: str) -> str:
    """Pull Python code out of a model response, handling markdown fences and prose."""
    # Try fenced code blocks first
    fence_match = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if fence_match:
        return fence_match.group(1).strip()
    # Fall back to: assume the whole thing is code, but strip leading prose lines
    lines = text.strip().split("\n")
    # Find first line that looks like code (def/class/import/assignment)
    for i, line in enumerate(lines):
        if re.match(r"^(def |class |import |from |\w+\s*=)", line.strip()):
            return "\n".join(lines[i:])
    return text.strip()


def robust_rubric(model_response: str, problem: dict) -> dict:
    """Same as rubric but extracts code first."""
    code = extract_code(model_response)
    results = []
    for test_call, expected in problem["tests"]:
        result = run_code(code, test_call, expected)
        results.append((test_call, result["passed"], result.get("error")))
    pass_rate = sum(1 for _, passed, _ in results if passed) / len(results)
    return {"pass_rate": pass_rate, "results": results, "extracted": code}


# Re-score the chatty output
print("Score with robust rubric:")
print(robust_rubric(chatty_code, PROBLEMS[0]))


Score with robust rubric:
{'pass_rate': 1.0, 'results': [('add(1, 2)', True, None), ('add(-5, 5)', True, None), ('add(0, 0)', True, None), ('add(100, 200)', True, None)], 'extracted': 'def add(a, b):\n  """\n  This function takes two numbers as input and returns their sum.\n\n  Args:\n    a: The first number.\n    b: The second number.\n\n  Returns:\n    The sum of a and b.\n  """\n  return a + b'}


**Lesson learned.** Real environments need **parsers** — components that pull structured information out of unstructured model output. Verifiers ships these built-in (`vf.XMLParser`, `vf.ThinkParser`, custom parser classes) precisely because rubric brittleness is the #1 footgun.

Generalizable principle: every reward function in production has two parts — *parse* the response, then *score* the parsed result. Conflating them is how you get bad RL training runs.

---

## 6. Port to Verifiers

Now let's rebuild the same environment using the [Verifiers library](https://github.com/PrimeIntellect-ai/verifiers) — Will Brown's library that powers Prime Intellect's training stack and was used to train the 100B+ INTELLECT-3 model.

What we get for free:
- **Parser classes** — built-in extraction (XMLParser, ThinkParser, custom)
- **Rubric class** — composable reward functions with weights
- **Environment classes** — `SingleTurnEnv`, `MultiTurnEnv`, `ToolEnv`, `CodeEnv`
- **Evaluation harness** — `env.evaluate(client, model, num_examples)` returns aggregated scores
- **Hub integration** — `prime env push` to share, `prime env install owner/name` to consume
- **Training compatibility** — same env runs in `prime-rl` for actual GRPO training


In [12]:
!pip install -q verifiers


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.8/645.8 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 826.2/826.2 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.0/841.0 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 727.0/727.0 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 95.3 MB/s eta 0:

In [13]:
import verifiers as vf
from datasets import Dataset
print(f"Verifiers version: {vf.__version__ if hasattr(vf, '__version__') else 'installed'}")


Verifiers version: 0.1.14


### Step 1: Convert problems into a HuggingFace Dataset

Verifiers expects a `Dataset` with at least a `question` (or `prompt`) column. We'll also stash the test cases in an `info` column so our reward function can find them.

In [18]:
import json

dataset_rows = []
for p in PROBLEMS:
    dataset_rows.append({
        "question": p["prompt"],
        "answer": p["function_name"],
        "info": {
            "function_name": p["function_name"],
            "tests_json": json.dumps(p["tests"]),
            "id": p["id"],
        },
    })

vf_dataset = Dataset.from_list(dataset_rows)
print(vf_dataset)
print("\nFirst row:")
print(vf_dataset[0])

Dataset({
    features: ['question', 'answer', 'info'],
    num_rows: 5
})

First row:
{'question': 'Write a Python function `add(a, b)` that returns a + b.', 'answer': 'add', 'info': {'function_name': 'add', 'id': 'add', 'tests_json': '[["add(1, 2)", 3], ["add(-5, 5)", 0], ["add(0, 0)", 0], ["add(100, 200)", 300]]'}}


### Step 2: Define the rubric

A Verifiers reward function takes `(prompt, completion, answer, state)` and returns a float. We'll define one that runs the tests, plus a second one that scores formatting (did the model produce parseable code?). Combining them shows weighted-rubric design.

In [19]:
def code_passes_tests(prompt, completion, answer, state, info=None, **kwargs) -> float:
    """Reward = fraction of tests passed. Range [0, 1]."""
    response = completion[-1]["content"] if isinstance(completion, list) else str(completion)
    code = extract_code(response)
    if info is None:
        return 0.0
    tests = info["tests"]
    passed = 0
    for test_call, expected in tests:
        r = run_code(code, test_call, expected)
        if r["passed"]:
            passed += 1
    return passed / len(tests)


def code_is_parseable(prompt, completion, answer, state, info=None, **kwargs) -> float:
    """Reward = 1.0 if the extracted code parses as Python, else 0."""
    response = completion[-1]["content"] if isinstance(completion, list) else str(completion)
    code = extract_code(response)
    try:
        compile(code, "<rubric>", "exec")
        return 1.0
    except SyntaxError:
        return 0.0


# Compose with weights: correctness matters 5x more than formatting
rubric = vf.Rubric(
    funcs=[code_passes_tests, code_is_parseable],
    weights=[5.0, 1.0],
)
print("Rubric created with 2 reward functions.")


Rubric created with 2 reward functions.


### Step 3: Build the SingleTurnEnv

This is the core API. Three lines.

In [20]:
SYSTEM_PROMPT = (
    "You are a Python coding assistant. Write a function that solves the user's request. "
    "Respond with the function definition only — no markdown fences, no explanation."
)

env = vf.SingleTurnEnv(
    dataset=vf_dataset,
    system_prompt=SYSTEM_PROMPT,
    rubric=rubric,
)
print("Environment built:", type(env).__name__)


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Environment built: SingleTurnEnv


### Step 4: Evaluate

`env.evaluate()` runs rollouts against any OpenAI-compatible client and returns aggregated scores. Same loop we wrote by hand, but with retries, async parallelism, and proper score aggregation built in.

In [31]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

# sanity check
assert os.environ["OPENROUTER_API_KEY"].startswith("sk-or-"), "key not set"

from verifiers.clients import ClientConfig

vf_config = ClientConfig(
    client_type="openai_chat_completions",
    api_key_var="OPENROUTER_API_KEY",
    api_base_url="https://openrouter.ai/api/v1",
)

results = await env.evaluate(
    client=vf_config,
    model="google/gemini-2.0-flash-001",
    num_examples=5,
    rollouts_per_example=1,
    sampling_args={"max_tokens": 400},
)

print(results)

eval_dataset is not set, falling back to train dataset
Processing 5 groups (5 total rollouts): 100%|██████████| 5/5 [00:01<00:00,  3.45it/s, reward=1]

{'outputs': [{'example_id': 0, 'prompt': [SystemMessage(role='system', content="You are a Python coding assistant. Write a function that solves the user's request. Respond with the function definition only — no markdown fences, no explanation."), UserMessage(role='user', content='Write a Python function `add(a, b)` that returns a + b.')], 'completion': [AssistantMessage(role='assistant', content='```python\ndef add(a, b):\n  """\n  This function adds two numbers a and b and returns the sum.\n  """\n  return a + b\n```\n', reasoning_content=None, thinking_blocks=None, tool_calls=None)], 'answer': 'add', 'info': {'function_name': 'add', 'id': 'add', 'tests_json': '[["add(1, 2)", 3], ["add(-5, 5)", 0], ["add(0, 0)", 0], ["add(100, 200)", 300]]'}, 'reward': 1.0, 'error': None, 'timing': {'start_time': 1778182946.090946, 'setup': {'start': 1778182946.090989, 'end': 1778182946.090991, 'duration': 1.9073486328125e-06}, 'generation': {'start': 1778182946.0909836, 'end': 1778182946.896391, 'dur

Same task, same data, ~30 lines instead of ~100. Plus we now have:

- A **publishable artifact** — `prime env push` would put this on the [Hub](https://app.primeintellect.ai/dashboard/environments).
- **Training compatibility** — pass `env` to `prime-rl` or Tinker and run GRPO with no further work.
- **Synthetic data extraction** — `env.make_dataset(results)` would turn high-reward rollouts into SFT data automatically.

This is what "build once, use many ways" means in practice.

---

## 7. Multi-turn — let the model iterate

Single-turn is the simplest case. The real magic of RL environments is **multi-turn**: the model gets feedback on its output and tries again. This is what makes coding agents work — write code, see test failures, fix.

We'll build the multi-turn loop **from scratch** first (so the mechanics are clear), then point at the Verifiers `MultiTurnEnv` API for production use.

The loop:

```
Turn 1:  user prompt          →  model writes code  →  run tests
                                                          │
                                       if all pass: done  │
                                       else: feedback ────┘
Turn 2:  prior context + feedback  →  model fixes code  →  run tests
                                                              │
                                                              ↓
                                                            ...
```


In [32]:
def multi_turn_rollout(
    problem: dict,
    model: str = "google/gemini-2.0-flash-001",
    max_turns: int = 3,
) -> dict:
    """Iterative coding loop: model writes -> we run tests -> we feed back errors -> repeat."""
    messages = [
        {"role": "system", "content": (
            "You are a Python coding assistant. Write a function that solves the user's request. "
            "If the user reports test failures, analyze them and produce a corrected function."
        )},
        {"role": "user", "content": problem["prompt"]},
    ]

    turn_history = []

    for turn in range(1, max_turns + 1):
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.0,
            max_tokens=400,
        )
        model_output = resp.choices[0].message.content
        messages.append({"role": "assistant", "content": model_output})

        code = extract_code(model_output)
        results = []
        for test_call, expected in problem["tests"]:
            r = run_code(code, test_call, expected)
            results.append((test_call, r["passed"], r.get("error")))

        n_pass = sum(1 for _, ok, _ in results if ok)
        pass_rate = n_pass / len(results)
        turn_history.append({"turn": turn, "pass_rate": pass_rate, "code": code})

        if pass_rate == 1.0:
            return {"final_pass_rate": 1.0, "turns_used": turn, "history": turn_history}

        # Build feedback message — this is the environment talking back
        failures = [(call, err) for call, ok, err in results if not ok]
        feedback_lines = [f"Your code passed {n_pass}/{len(results)} tests. Failures:"]
        for call, err in failures[:3]:   # cap at 3 to avoid context bloat
            feedback_lines.append(f"  - {call} → {err}")
        feedback_lines.append("Please fix and provide the corrected function.")
        messages.append({"role": "user", "content": "\n".join(feedback_lines)})

    return {"final_pass_rate": pass_rate, "turns_used": max_turns, "history": turn_history}


Let's deliberately give the model a problem it might get wrong on the first try and watch it iterate.

In [33]:
# Pick a problem and add an extra-tricky test to provoke a first-turn failure
tricky_problem = {
    "id": "edge_count_vowels",
    "prompt": (
        "Write a Python function `count_vowels(s)` that returns the count of vowels in s. "
        "Vowels are aeiouAEIOU. The function must also handle the case where 'y' is a vowel "
        "if and only if it is NOT at the start or end of the string."
    ),
    "function_name": "count_vowels",
    "tests": [
        ("count_vowels('hello')", 2),
        ("count_vowels('rhythm')", 1),       # y in the middle, counts
        ("count_vowels('yellow')", 2),       # y at start, doesn't count, but e/o do
        ("count_vowels('hairy')", 2),        # y at end, doesn't count, but a/i do
        ("count_vowels('lyly')", 1),         # only middle y counts
    ],
}

print(f"Running multi-turn rollout (max 3 turns) on: {tricky_problem['id']}")
print("=" * 60)
trace = multi_turn_rollout(tricky_problem, max_turns=3)

for h in trace["history"]:
    print(f"\n--- Turn {h['turn']}: pass rate {h['pass_rate']:.0%} ---")
    print(h["code"][:400])

print("\n" + "=" * 60)
print(f"Final: {trace['final_pass_rate']:.0%} after {trace['turns_used']} turn(s)")


Running multi-turn rollout (max 3 turns) on: edge_count_vowels

--- Turn 1: pass rate 100% ---
def count_vowels(s):
    """Counts the number of vowels (aeiouAEIOU) in the string s.
    'y' is a vowel if and only if it is NOT at the start or end of the string.
    """
    vowels = "aeiouAEIOU"
    count = 0
    for i, char in enumerate(s):
        if char in vowels:
            count += 1
        elif char in "yY" and 0 < i < len(s) - 1:
            count += 1
    return count

Final: 100% after 1 turn(s)


In [ ]:
tricky_problem = {
    "id": "edge_count_vowels_v2",
    "prompt": (
        "Write `count_vowels(s)` that returns the count of vowels in s. "
        "Rules: vowels are aeiouAEIOU. 'y' is a vowel ONLY when NOT at the start or end. "
        "If s contains ANY digit, return -1. If s is None, return 0 (not an error). "
        "Apostrophes count as silent — skip them entirely (don't break adjacency for the y rule)."
    ),
    "function_name": "count_vowels",
    "tests": [
        ("count_vowels('hello')", 2),
        ("count_vowels('rhythm')", 1),
        ("count_vowels('yellow')", 2),
        ("count_vowels('hairy')", 2),
        ("count_vowels('hello123')", -1),
        ("count_vowels(None)", 0),
        ("count_vowels(\"y'all\")", 0),
    ],
}

**Watch the pass rate climb (or not).** This is the rollout structure that GRPO actually optimizes during RL training — the trajectory is "code → feedback → code → feedback → ...", and the reward is the final pass rate (or some shaped function of pass rate + turn count).

A few design questions worth pausing on:

- **Reward shaping.** Should we reward "passes on turn 1" more than "passes on turn 3"? Yes if we care about efficiency, no if we're just optimizing correctness. This is a real choice in production.
- **Feedback quality.** Our environment passes raw test failures to the model. A more sophisticated environment might use an LLM to *interpret* the failures into a hint. Tradeoff: hint quality vs. dependency on a judge model.
- **Context bloat.** By turn 5 the conversation is long. Real harnesses do summarization or selective context compression. (Class 12 territory.)

### Verifiers equivalent

For production, you'd subclass `vf.MultiTurnEnv` instead of writing the loop by hand. The pattern looks like this — we won't run it (subclassing has version-specific details), but here's the shape:

```python
class CodeFixerEnv(vf.MultiTurnEnv):
    async def env_response(self, messages, state, **kwargs):
        # messages = full conversation including latest assistant turn
        # state = persistent dict (problem info, turn counter, etc.)
        latest = messages[-1]["content"]
        code = extract_code(latest)
        # ... run tests, build feedback ...
        if all_passed:
            return [], state  # empty -> end episode
        return [{"role": "user", "content": feedback}], state

    async def is_completed(self, messages, state, **kwargs):
        return state.get("all_passed", False) or state.get("turn", 0) >= self.max_turns
```

Plug that into the same `vf.SingleTurnEnv → MultiTurnEnv` swap and you get the same multi-turn behavior with all the Verifiers infrastructure (eval harness, training compat, hub publishability) for free.

---

## 8. The punchline — environments give you ground-truth model comparisons

Now the part that makes all this matter. We have an environment. We can run *any* model through it and get a comparable score. This is what "environment as eval" really means: an environment turns subjective model quality into a number.

Let's run three models on the same problems and rank them.


In [37]:
def score_code(generated_code: str, problem: dict) -> dict:
    code = extract_code(generated_code)   # ← add this
    results = []
    for test_call, expected in problem["tests"]:
        result = run_code(code, test_call, expected)
        results.append((test_call, result["passed"], result.get("error")))
    pass_rate = sum(1 for _, passed, _ in results if passed) / len(results)
    return {"pass_rate": pass_rate, "results": results}


def evaluate(problems, model):
    scores, transcripts = [], []
    for p in problems:
        code = rollout(p, model=model)
        s = score_code(code, p)
        scores.append(s["pass_rate"])
        transcripts.append({"problem": p["id"], "code": code, "pass_rate": s["pass_rate"], "details": s["results"]})
        print(f"  {p['id']}: {s['pass_rate']:.0%} ({sum(1 for _, ok, _ in s['results'] if ok)}/{len(s['results'])} tests)")
    return {"mean_pass_rate": sum(scores) / len(scores), "transcripts": transcripts}

In [38]:
# Three models of varying capability available on OpenRouter (free or low-cost tiers)
MODELS_TO_COMPARE = [
    "google/gemini-2.0-flash-001",
    "openai/gpt-4o-mini",
    "qwen/qwen-2.5-coder-32b-instruct",
]

comparison = {}
for model_id in MODELS_TO_COMPARE:
    print(f"\n{'=' * 50}")
    print(f"Evaluating: {model_id}")
    print('=' * 50)
    try:
        result = evaluate(PROBLEMS, model=model_id)
        comparison[model_id] = result["mean_pass_rate"]
    except Exception as e:
        print(f"  ERROR: {e}")
        comparison[model_id] = None

print("\n\n" + "=" * 50)
print("FINAL RANKING")
print("=" * 50)
ranked = sorted(
    [(m, s) for m, s in comparison.items() if s is not None],
    key=lambda x: -x[1]
)
for i, (model, score) in enumerate(ranked, 1):
    print(f"  {i}. {model}: {score:.1%}")



Evaluating: google/gemini-2.0-flash-001
  add: 100% (4/4 tests)
  is_even: 100% (4/4 tests)
  reverse_string: 100% (4/4 tests)
  fizzbuzz: 100% (4/4 tests)
  count_vowels: 100% (4/4 tests)

Evaluating: openai/gpt-4o-mini
  add: 100% (4/4 tests)
  is_even: 100% (4/4 tests)
  reverse_string: 100% (4/4 tests)
  fizzbuzz: 0% (0/4 tests)
  count_vowels: 0% (0/4 tests)

Evaluating: qwen/qwen-2.5-coder-32b-instruct
  add: 100% (4/4 tests)
  is_even: 0% (0/4 tests)
  reverse_string: 0% (0/4 tests)
  fizzbuzz: 0% (0/4 tests)
  count_vowels: 0% (0/4 tests)


FINAL RANKING
  1. google/gemini-2.0-flash-001: 100.0%
  2. openai/gpt-4o-mini: 60.0%
  3. qwen/qwen-2.5-coder-32b-instruct: 20.0%


**This is the most important slide of the class.**

Three models. Same five problems. Same scoring. Different numbers. The rankings are now *comparable* in a way that "vibes" or "I tried it on a few examples" never can be.

Now imagine:
- Replace 5 toy problems with 1000 real coding tasks → you have a benchmark.
- Replace test-passing rubric with task completion in a sandboxed git repo → you have **SWE-bench**.
- Use the rubric scores as **rewards** instead of just metrics → you have RL training.
- Filter to only the high-reward rollouts → you have **synthetic SFT data**.

Same artifact. Different uses. That's the Will Brown insight, made concrete.

---

## 9. Wrap-up + what's next

### What you built
- A coding environment **from scratch** (~50 lines, no library)
- The same environment **in Verifiers** with composable rubrics and parsers
- A **multi-turn iterative** version where the model fixes its own bugs
- A **cross-model comparison** that turns subjective quality into a ranked number

### Key takeaways

1. **Environment = dataset + rollout + rubric.** No magic, three pieces.
2. **The same environment serves training, eval, and synthetic data.** That's why labs (and Prime Intellect's open-source counter-effort) are pouring resources into them.
3. **Rubric brittleness is the #1 footgun.** Always parse before scoring; never conflate the two.
4. **Multi-turn is where the action is.** Single-turn is for benchmarks; real RL training and real agents are multi-turn.
5. **Open environments are a public good.** Closed labs' proprietary environments are a moat. The [Environments Hub](https://app.primeintellect.ai/dashboard/environments) is the open community's response.

### References

- **Verifiers**: [github.com/PrimeIntellect-ai/verifiers](https://github.com/PrimeIntellect-ai/verifiers) · [docs](https://docs.primeintellect.ai/verifiers)
- **Environments Hub**: [app.primeintellect.ai/dashboard/environments](https://app.primeintellect.ai/dashboard/environments)
- **OpenEnv** (Meta + HF): [github.com/meta-pytorch/OpenEnv](https://github.com/meta-pytorch/OpenEnv)
- **Anakin87's Environments Hub walkthrough**: [hf.co/blog/anakin87/environments-hub](https://huggingface.co/blog/anakin87/environments-hub)
- **INTELLECT-3 technical report**: [arxiv.org/abs/2512.16144](https://arxiv.org/abs/2512.16144) — the credibility receipt
- **Will Brown on environments** (Verifiers paper): [arxiv link](https://github.com/PrimeIntellect-ai/verifiers#citation)
